# Model Selection

**Benchamarked Models (regression problem)**

- Explainable Boosting (GAM ensemble boosting)
https://interpret.ml/docs/python/api/ExplainableBoostingRegressor.html

- Random Forest (ensemble bagging)
https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html

- Extreme Gradient Boosting (ensemble boosting)
https://xgboost.readthedocs.io/en/stable/python/python_api.html

- Gradient Boosting (ensemble boosting)
https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html

- Super Vector (kernel)
https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html#sklearn.svm.SVR

- Elastic Net (regularized linear)
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html#sklearn.linear_model.ElasticNet

- Bayesian Ridge Regression (bayesian linear)
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.BayesianRidge.html

- Deep Learning (neural networks)
https://skorch.readthedocs.io/en/stable/regressor.html


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import itertools
import os
import time

In [2]:
# Reegression models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost.sklearn import XGBRegressor
from sklearn.svm import SVR
from sklearn.linear_model import ElasticNet
from sklearn import linear_model
from interpret.glassbox import ExplainableBoostingRegressor
from sklearn.metrics import mean_squared_error

In [3]:
sns.set_theme(style="darkgrid")
sns.set_palette('tab10')

In [4]:
seed=112358

In [5]:
df_projects=pd.read_csv(os.path.join('results','model_selection_projects.csv'))
df_selection=df_projects[df_projects['ModelSelection']==True]
df_selection[['Project','SP']].groupby('SP').count()

,Project
SP,
0.068966,10
0.172414,10
0.275862,10
0.379310,10
0.482759,10
0.586207,10
0.689655,10
0.793103,10
0.896552,10


## Random Forest

In [1]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# Hyperparameters of Random Forest
hyperparam={"n_estimators": [100,500,1000],
            "max_depth": [1,5,10]}

In [17]:
pd.read_csv(os.path.join('results',"test_runs.csv"))

,test_runs
0,8755
1,1064
2,2099
3,7581
4,7410
...,...
1995,3424
1996,1400
1997,2428
1998,7704


In [ ]:
mdl_name="rfr"
mse_test = []
for sp in df_selection['SP'].unique():

    print(f"Runing SP: {sp}...")

    # List of projects for training and testing
    for project in df_selection.loc[df_selection['SP'] == sp,'Project']:
        df = pd.read_csv(os.path.join('results',f"{project}_simulation_e1.csv"),index_col=0)

        # Read test runs
        df_runs= pd.read_csv(os.path.join('results',"test_runs.csv"))

        # Training data
        df_train=df.loc[df.index.difference(df_runs['test_runs'].values)]
        Xtrain=df_train.iloc[:,:32]
        Ytrain=df_train.iloc[:,-1]
        
        # Testing data
        df_test=df.loc[df_runs['test_runs'].values]
        Xtest=df_test.iloc[:,:32]
        Ytest=df_test.iloc[:,-1]

        # Test all combinations of hiperparameters
        for combination in itertools.product(hyperparam['n_estimators'], hyperparam['max_depth']):
            # Train the model
            model = RandomForestRegressor(n_estimators=combination[0], max_depth=combination[1],n_jobs=-1,random_state=seed)
            model.fit(Xtrain,Ytrain)
            # Predict the test set
            Ypred=model.predict(Xtest)
            # Calculate the MSE
            mse = mean_squared_error(Ytest, Ypred)
            # Update the test runs
            mse_test.append([sp,project,combination[0],combination[1],mse])
# Save the results of the model selection
pd.DataFrame(mse_test,columns=['SP','Project','n_estimators','max_depth','MSE']).to_csv(os.path.join('results',f"model_selection_{mdl_name}.csv"),index=False)


Runing SP: 0.1724137931034483...
Runing SP: 0.0689655172413793...
Runing SP: 0.2758620689655172...
Runing SP: 0.3793103448275862...
Runing SP: 0.4827586206896552...
Runing SP: 0.5862068965517241...
Runing SP: 0.6896551724137931...
Runing SP: 0.7931034482758621...
Runing SP: 0.896551724137931...


In [6]:
df_rf=pd.read_csv(os.path.join('results',f"model_selection_rfr.csv"))

In [7]:
results=df_rf.groupby(['SP','n_estimators','max_depth']).agg({'MSE':'mean'}).reset_index()
for sp in results['SP'].unique():
    df_sp=results[results['SP']==sp].sort_values(by='MSE')
    print(f"SP: {sp} - Best hiperparameters: n_estimators={df_sp.iloc[0,1]} and max_depth={df_sp.iloc[0,2]}")

SP: 0.0689655172413793 - Best hiperparameters: n_estimators=1000 and max_depth=10
SP: 0.1724137931034483 - Best hiperparameters: n_estimators=1000 and max_depth=10
SP: 0.2758620689655172 - Best hiperparameters: n_estimators=1000 and max_depth=10
SP: 0.3793103448275862 - Best hiperparameters: n_estimators=1000 and max_depth=10
SP: 0.4827586206896552 - Best hiperparameters: n_estimators=500 and max_depth=10
SP: 0.5862068965517241 - Best hiperparameters: n_estimators=1000 and max_depth=10
SP: 0.6896551724137931 - Best hiperparameters: n_estimators=1000 and max_depth=10
SP: 0.7931034482758621 - Best hiperparameters: n_estimators=1000 and max_depth=10
SP: 0.896551724137931 - Best hiperparameters: n_estimators=1000 and max_depth=10


In [86]:
hyperparam={"n_estimators": [101,550,1000],
            "max_depth": [1,5,14]}
mdl=GradientBoostingRegressor
params=list(hyperparam.keys())
values=list(hyperparam.values())
params=list(hp.keys())
for val in itertools.product(values[0],values[1]):
    m1=mdl()
    m1.set_params(**{params[0]:val[0],params[1]:val[1]})
    print(m1.get_params())


{'alpha': 0.9, 'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'learning_rate': 0.1, 'loss': 'squared_error', 'max_depth': 1, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 101, 'n_iter_no_change': None, 'random_state': None, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}
{'alpha': 0.9, 'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'learning_rate': 0.1, 'loss': 'squared_error', 'max_depth': 5, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 101, 'n_iter_no_change': None, 'random_state': None, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}
{'alpha': 0.9, 'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'lea